# Test: Seasonal Forcing Ensemble Experiment

Quick validation that every component loads, configs resolve, forcing data
is readable, parameters exist, and ensemble forcing files can be built
for a **single donor year / single season** without running SUMMA.

Run all cells. Every cell should print `PASS` at the end.

## Test 0 — Imports

In [1]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'ess-project' / 'modeling'))

from seasonal_ensemble_experiment import (
    SeasonalEnsembleExperiment,
    ExperimentConfig,
    ParameterOptimizer,
    ModelRunner,
    ForcingEnsembleBuilder,
    EnsembleRunner,
    ParallelEnsembleRunner,
    EnsembleEvaluator,
    EnsembleVisualizer,
    SEASONS,
    FORCING_VARS,
)

assert len(SEASONS) == 4
assert len(FORCING_VARS) == 7
print(f"Seasons: {list(SEASONS.keys())}")
print(f"Forcing vars: {FORCING_VARS}")
print("PASS: all imports OK")

Seasons: ['DJF', 'MAM', 'JJA', 'SON']
Forcing vars: ['pptrate', 'SWRadAtm', 'LWRadAtm', 'airpres', 'airtemp', 'windspd', 'spechum']
PASS: all imports OK


## Test 1 — ExperimentConfig loads and paths resolve

In [2]:
CONFIG_PATH = str(PROJECT_ROOT / 'ess-project' / '0_config_files' / 'config_East_River_lumped.yaml')
assert Path(CONFIG_PATH).exists(), f"Config not found: {CONFIG_PATH}"

cfg = ExperimentConfig(CONFIG_PATH)

assert cfg.domain_name == 'East_River_lumped'
assert cfg.project_dir.exists(), f"Project dir missing: {cfg.project_dir}"
assert cfg.settings_dir.exists(), f"SUMMA settings missing: {cfg.settings_dir}"
assert cfg.summa_input_dir.exists(), f"Forcing dir missing: {cfg.summa_input_dir}"

print(f"Domain:       {cfg.domain_name}")
print(f"Project dir:  {cfg.project_dir}")
print(f"Settings:     {cfg.settings_dir}")
print(f"Forcing:      {cfg.summa_input_dir}")
print(f"Opt dir:      {cfg.opt_dir}")
print(f"Ensemble dir: {cfg.ensemble_dir}")
print("PASS: config loads, all key paths exist")

Domain:       East_River_lumped
Project dir:  /scratch/dlhogan/ess-project-data/domain_East_River_lumped
Settings:     /scratch/dlhogan/ess-project-data/domain_East_River_lumped/settings/SUMMA
Forcing:      /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/SUMMA_input
Opt dir:      /scratch/dlhogan/ess-project-data/domain_East_River_lumped/optimisation
Ensemble dir: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/seasonal_ensemble
PASS: config loads, all key paths exist


## Test 2 — Best parameters can be loaded

In [3]:
optimizer = ParameterOptimizer(cfg)
best_params = optimizer.get_best_parameters()

assert isinstance(best_params, pd.DataFrame)
assert len(best_params) > 0, "No parameters loaded"
assert 'parameter' in best_params.columns
assert 'value' in best_params.columns

print(f"Loaded {len(best_params)} parameters:")
for _, row in best_params.iterrows():
    print(f"  {row['parameter']:30s} = {row['value']}")
print("PASS: parameters loaded")

2026-03-09 15:21:27,260 - SeasonalEnsemble - INFO - Loading best parameters from /scratch/dlhogan/ess-project-data/domain_East_River_lumped/optimisation/dds_bigBuckt_20260227/best_parameters.csv


Loaded 12 parameters:
  k_soil                         = 9.4e-06
  theta_sat                      = 0.516
  theta_res                      = 0.027
  aquiferScaleFactor             = 50.0
  rootingDepth                   = 6.876
  critSoilWilting                = 0.075
  critSoilTranspire              = 0.175
  frozenPrecipMultip             = 1.0
  basin__aquiferScaleFactor      = 50.0
  basin__aquiferHydCond          = 0.001
  routingGammaShape              = 2.5
  routingGammaScale              = 46000.0
PASS: parameters loaded


## Test 3 — Forcing data is loadable and covers expected range

In [4]:
builder = ForcingEnsembleBuilder(cfg)

# Load just the first and last files to check date range without loading everything
forcing_files = sorted(cfg.summa_input_dir.glob('*.nc'))
assert len(forcing_files) > 0, "No forcing files found"

first_ds = xr.open_dataset(forcing_files[0])
last_ds = xr.open_dataset(forcing_files[-1])

first_time = pd.Timestamp(first_ds.time.values[0])
last_time = pd.Timestamp(last_ds.time.values[-1])
first_ds.close()
last_ds.close()

print(f"Forcing files: {len(forcing_files)}")
print(f"Date range: {first_time} to {last_time}")

# Check that our experiment period is covered
assert first_time.year <= 1999, f"Forcing starts too late: {first_time}"
assert last_time.year >= 2019, f"Forcing ends too early: {last_time}"

# Check forcing variables in one file
sample = xr.open_dataset(forcing_files[0])
for var in FORCING_VARS:
    assert var in sample.data_vars, f"Missing forcing var: {var}"
sample.close()

print(f"All {len(FORCING_VARS)} forcing variables present")
print("PASS: forcing data OK")

Forcing files: 468
Date range: 1981-01-01 00:00:00 to 2019-12-31 23:00:00
All 7 forcing variables present
PASS: forcing data OK


## Test 4 — SUMMA settings files exist

In [5]:
required_files = [
    'fileManager.txt',
    'forcingFileList.txt',
    'modelDecisions.txt',
    'outputControl.txt',
    'localParamInfo.txt',
    'basinParamInfo.txt',
    'attributes.nc',
    'coldState.nc',
    'trialParams.nc',
]

missing = []
for fname in required_files:
    p = cfg.settings_dir / fname
    status = 'OK' if p.exists() else 'MISSING'
    if not p.exists():
        missing.append(fname)
    print(f"  {status:7s}  {fname}")

assert not missing, f"Missing SUMMA files: {missing}"
print("PASS: all SUMMA settings files present")

  OK       fileManager.txt
  OK       forcingFileList.txt
  OK       modelDecisions.txt
  OK       outputControl.txt
  OK       localParamInfo.txt
  OK       basinParamInfo.txt
  OK       attributes.nc
  OK       coldState.nc
  OK       trialParams.nc
PASS: all SUMMA settings files present


## Test 5 — Observations file exists and is parseable

In [6]:
obs_files = list(cfg.obs_dir.glob('*streamflow*.csv'))
assert len(obs_files) > 0, f"No observation files in {cfg.obs_dir}"

obs_df = pd.read_csv(obs_files[0], parse_dates=['datetime'])
assert 'datetime' in obs_df.columns
assert 'discharge_cms' in obs_df.columns

print(f"Obs file: {obs_files[0].name}")
print(f"Obs range: {obs_df['datetime'].min()} to {obs_df['datetime'].max()}")
print(f"Obs records: {len(obs_df)}")
print("PASS: observations loadable")

Obs file: East_River_lumped_streamflow_processed.csv
Obs range: 2007-10-01 00:00:00 to 2026-03-08 00:00:00
Obs records: 161593
PASS: observations loadable


## Test 6 — Build one ensemble forcing file (single season, single donor)

In [7]:
import tempfile, shutil

# Use a temp dir so we don't pollute the project
test_dir = Path(tempfile.mkdtemp(prefix='ensemble_test_'))

# Load a subset of forcing: just 2010 and 2019 (donor + target)
target_year = 2019
donor_years = [2010]
season = 'MAM'  # spring

# Load only the months we need to keep this fast
needed_files = []
for f in sorted(cfg.summa_input_dir.glob('*.nc')):
    # Extract YYYYMM from filename
    parts = f.stem.split('_')
    yyyymm = parts[-1]  # e.g., 201903
    year = int(yyyymm[:4])
    if year in [2010, 2019]:
        needed_files.append(f)

print(f"Loading {len(needed_files)} forcing files for years 2010 and 2019...")
subset_ds = xr.open_mfdataset(needed_files, combine='by_coords')

# Build one ensemble member
created = builder.build_ensemble_for_season(
    full_forcing=subset_ds,
    target_year=target_year,
    donor_years=donor_years,
    season=season,
    output_dir=test_dir / season,
)
subset_ds.close()

assert len(created) == 1, f"Expected 1 file, got {len(created)}"
ensemble_file = created[0]
assert ensemble_file.exists()

# Validate the ensemble file
ens_ds = xr.open_dataset(ensemble_file)
ens_times = pd.DatetimeIndex(ens_ds.time.values)
print(f"\nEnsemble file: {ensemble_file.name}")
print(f"Time range: {ens_times[0]} to {ens_times[-1]}")
print(f"Timesteps: {len(ens_times)}")
print(f"Variables: {list(ens_ds.data_vars)}")

# Check all forcing vars are present
for var in FORCING_VARS:
    assert var in ens_ds.data_vars, f"Missing var in ensemble file: {var}"

# Check the spring months have data from 2010 (not the same as 2019 original)
spring_mask = ens_times.month.isin([3, 4, 5])
assert spring_mask.any(), "No spring timesteps found"
print(f"Spring timesteps: {spring_mask.sum()}")

ens_ds.close()

# Cleanup
shutil.rmtree(test_dir)
print("PASS: ensemble forcing file built and validated")

Loading 24 forcing files for years 2010 and 2019...


2026-03-09 15:21:54,583 - SeasonalEnsemble - INFO -   Building ensemble: MAM from 2010 → 2019
2026-03-09 15:31:07,923 - SeasonalEnsemble - INFO -     Saved: East_River_lumped_MAM_2019_from_2010.nc



Ensemble file: East_River_lumped_MAM_2019_from_2010.nc
Time range: 2019-01-01 00:00:00 to 2019-12-31 23:00:00
Timesteps: 8760
Variables: ['latitude', 'longitude', 'hruId', 'airpres', 'LWRadAtm', 'SWRadAtm', 'pptrate', 'airtemp', 'spechum', 'windspd', 'data_step']
Spring timesteps: 2208
PASS: ensemble forcing file built and validated


## Test 7 — ParallelEnsembleRunner workspace preparation

In [8]:
# Test that prepare_member creates the right directory structure
# without actually running SUMMA

test_dir = Path(tempfile.mkdtemp(prefix='parallel_test_'))

# Temporarily override ensemble_dir
orig_ensemble_dir = cfg.ensemble_dir
cfg.ensemble_dir = test_dir

runner = ParallelEnsembleRunner(cfg, max_workers=2)

# Create a dummy forcing file
forcing_dir = test_dir / 'forcing' / 'MAM'
forcing_dir.mkdir(parents=True)
dummy_forcing = forcing_dir / 'East_River_lumped_MAM_2019_from_2010.nc'
dummy_forcing.write_text('')  # placeholder

mid = runner.prepare_member(
    season='MAM', donor_year=2010, target_year=2019, forcing_file=dummy_forcing
)

assert mid == 'MAM_from2010'
assert mid in runner.members

info = runner.members[mid]
assert info['run_dir'].exists(), f"Run dir not created: {info['run_dir']}"
assert info['fm_path'].exists(), f"fileManager not created: {info['fm_path']}"
assert info['output_dir'].exists(), f"Output dir not created: {info['output_dir']}"

# Check member-specific fileManager.txt contents
fm_text = info['fm_path'].read_text()
assert 'MAM_from2010' in fm_text or 'East_River_lumped_MAM_from2010' in fm_text, \
    f"outFilePrefix not set in fileManager:\n{fm_text}"

print(f"Member ID: {mid}")
print(f"Run dir: {info['run_dir']}")
print(f"Output dir: {info['output_dir']}")
print(f"Log file: {info['log_file']}")
print(f"Prefix: {info['prefix']}")

# Restore and cleanup
cfg.ensemble_dir = orig_ensemble_dir
shutil.rmtree(test_dir)
print("PASS: parallel workspace setup works")

Member ID: MAM_from2010
Run dir: /tmp/parallel_test_xpfvzsxx/runs/MAM_from2010
Output dir: /tmp/parallel_test_xpfvzsxx/results/MAM/from_2010
Log file: /tmp/parallel_test_xpfvzsxx/logs/MAM_from2010.log
Prefix: East_River_lumped_MAM_from2010
PASS: parallel workspace setup works


## Test 8 — Shell script generation

In [9]:
test_dir = Path(tempfile.mkdtemp(prefix='script_test_'))
orig_ensemble_dir = cfg.ensemble_dir
cfg.ensemble_dir = test_dir

runner = ParallelEnsembleRunner(cfg, max_workers=2)

# Create dummy forcing files for 2 members
for donor in [2010, 2011]:
    forcing_dir = test_dir / 'forcing' / 'MAM'
    forcing_dir.mkdir(parents=True, exist_ok=True)
    dummy = forcing_dir / f'East_River_lumped_MAM_2019_from_{donor}.nc'
    dummy.write_text('')
    runner.prepare_member('MAM', donor, 2019, dummy)

# Generate script
script_path = runner.generate_run_script()
assert script_path.exists()

script_text = script_path.read_text()
assert '#!/bin/bash' in script_text
assert 'MAX_PARALLEL=2' in script_text
assert 'MAM_from2010' in script_text
assert 'MAM_from2011' in script_text
assert 'ENSEMBLE COMPLETE' in script_text

# Check script is executable
assert os.access(script_path, os.X_OK)

print(f"Script: {script_path}")
print(f"Script size: {script_path.stat().st_size} bytes")
print(f"Lines: {len(script_text.splitlines())}")
print(f"\nFirst 5 lines:")
for line in script_text.splitlines()[:5]:
    print(f"  {line}")

cfg.ensemble_dir = orig_ensemble_dir
shutil.rmtree(test_dir)
print("\nPASS: script generation works")

2026-03-09 15:32:30,382 - SeasonalEnsemble - INFO - Run script: /tmp/script_test_olz4wuwd/run_ensemble.sh
2026-03-09 15:32:30,384 - SeasonalEnsemble - INFO -   Overnight usage:  nohup bash /tmp/script_test_olz4wuwd/run_ensemble.sh > /tmp/script_test_olz4wuwd/logs/ensemble_main.log 2>&1 &


Script: /tmp/script_test_olz4wuwd/run_ensemble.sh
Script size: 2438 bytes
Lines: 85

First 5 lines:
  #!/bin/bash
  # Seasonal Forcing Ensemble — auto-generated run script
  # Generated: 2026-03-09T15:32:30.382671
  # Members:   2
  # Max parallel: 2

PASS: script generation works


## Test 9 — Evaluator and Visualizer instantiate correctly

In [10]:
evaluator = EnsembleEvaluator(cfg)
assert evaluator.cfg.domain_name == 'East_River_lumped'

visualizer = EnsembleVisualizer(cfg)
assert visualizer.cfg.plots_dir.parent.exists()  # parent should exist

print("PASS: evaluator and visualizer instantiate")

PASS: evaluator and visualizer instantiate


## Test 10 — Full orchestrator initializes

In [ ]:
CONFIG_PATH = str(PROJECT_ROOT / 'ess-project' / '0_config_files' / 'config_East_River_lumped.yaml')
exp = SeasonalEnsembleExperiment(CONFIG_PATH, max_workers=4)

assert exp.cfg.domain_name == 'East_River_lumped'
assert exp.optimizer is not None
assert exp.ensemble_builder is not None
assert exp.ensemble_runner is not None
assert exp.evaluator is not None
assert exp.visualizer is not None
assert exp.max_workers == 4

print(f"Orchestrator ready for domain: {exp.cfg.domain_name}")
print(f"Max workers: {exp.max_workers}")
print("PASS: orchestrator initializes")

Orchestrator ready for domain: East_River_lumped
Max workers: 4

ALL TESTS PASSED


## Test 11 — Data coverage check (forcing + observations)

In [12]:
# Check that forcing + obs cover the full experiment period
BASELINE_START_YEAR = 1999
TARGET_YEAR = 2019

report = exp.check_data_coverage(BASELINE_START_YEAR, TARGET_YEAR)

print(f"\n{'=' * 60}")
print("DATA COVERAGE REPORT")
print(f"{'=' * 60}")
print(f"  Experiment period: {BASELINE_START_YEAR} to {TARGET_YEAR}")
print(f"  Forcing complete:  {'YES' if report['forcing_ok'] else 'NO'}")
print(f"  Forcing years OK:  {len(report['forcing_years'])}/{TARGET_YEAR - BASELINE_START_YEAR + 1}")
if report['forcing_gaps']:
    print(f"  Forcing gaps:      {report['forcing_gaps']}")

print(f"  Obs available:     {report['obs_start']} to {report['obs_end']}")
print(f"  Obs complete:      {'YES' if report['obs_ok'] else 'NO'}")
if report['obs_gap_years']:
    print(f"  Obs gap years:     {report['obs_gap_years']}")
    print(f"  NOTE: Obs gaps only affect evaluation plots, not the ensemble runs")

# Forcing MUST be complete; obs gaps are tolerable
assert report['forcing_ok'], f"Forcing has gaps: {report['forcing_gaps']}"
print("\nPASS: forcing covers full experiment period")

if not report['obs_ok']:
    print(f"WARNING: obs starts {report['obs_start'].year}, "
          f"missing {len(report['obs_gap_years'])} years before that")
    print("  To extend obs, run:")
    print("    exp.step0_prepare_data(start='1999-01-01 00:00', download_obs=True, create_forcing=False)")
else:
    print("PASS: obs also covers full period")

2026-03-09 15:54:45,293 - SeasonalEnsemble - INFO - 
2026-03-09 15:54:45,295 - SeasonalEnsemble - INFO - DATA COVERAGE CHECK
2026-03-09 15:54:45,296 - SeasonalEnsemble - INFO - ======================================================================
2026-03-09 15:54:45,301 - SeasonalEnsemble - INFO - Forcing: 21/21 years complete — ALL GOOD
2026-03-09 15:54:45,430 - SeasonalEnsemble - INFO - Observations: 2007-10-01 to 2026-03-08
2026-03-09 15:54:45,431 - SeasonalEnsemble - WARNING -   Obs missing for years: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006]
  (Obs not strictly required — only used for evaluation plots)



DATA COVERAGE REPORT
  Experiment period: 1999 to 2019
  Forcing complete:  YES
  Forcing years OK:  21/21
  Obs available:     2007-10-01 00:00:00 to 2026-03-08 00:00:00
  Obs complete:      NO
  Obs gap years:     [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006]
  NOTE: Obs gaps only affect evaluation plots, not the ensemble runs

PASS: forcing covers full experiment period
  To extend obs, run:
    exp.step0_prepare_data(start='1999-01-01 00:00', download_obs=True, create_forcing=False)


In [13]:
print("=" * 50)
print("ALL TESTS PASSED")
print("=" * 50)

ALL TESTS PASSED


---
## Running the Full Experiment Overnight with tmux

Once all tests pass, here's how to run the full experiment in `tmux` so it survives SSH disconnection:

### Option A: Python-managed parallel (tmux keeps the process alive)

```bash
# 1. Create a tmux session
tmux new -s ensemble

# 2. Activate the environment
conda activate CONFLUENCE-base

# 3. Run the experiment (blocks until complete)
cd /home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro/ess-project/modeling
python -c "
from seasonal_ensemble_experiment import SeasonalEnsembleExperiment
exp = SeasonalEnsembleExperiment(
    '../0_config_files/config_East_River_lumped.yaml',
    max_workers=12
)
exp.run_full_workflow(
    baseline_start='1999-01-02 01:00',
    baseline_end='2021-12-31 23:00',
    target_year=2020,
    skip_optimization=False,
    parallel=True,
    max_workers=12,
)
"

# 4. Detach from tmux: press Ctrl+B then D
# 5. Reconnect later:  tmux attach -t ensemble
```

### Option B: Shell script (nohup inside tmux for extra safety)

```bash
# 1. Create tmux session
tmux new -s ensemble

# 2. Activate env & prepare script from Python
conda activate CONFLUENCE-base
cd /home/dlhogan/projects/forked-repos/CONFLUENCE-uwmtnhydro/ess-project/modeling
python -c "
from seasonal_ensemble_experiment import SeasonalEnsembleExperiment
exp = SeasonalEnsembleExperiment('../0_config_files/config_East_River_lumped.yaml', max_workers=4)
# Load params and build forcing (fast steps)
exp.step1_optimize(skip_if_exists=True)
exp.step4_build_ensembles(target_year=2019, donor_years=list(range(1999, 2019)))
# Generate the bash run script
script = exp.step5_generate_script(target_year=2019, max_workers=4)
print(f'Script ready: {script}')
"

# 3. Launch the generated script
LOGS=/scratch/dlhogan/ess-project-data/domain_East_River_lumped/seasonal_ensemble/logs
nohup bash /scratch/dlhogan/ess-project-data/domain_East_River_lumped/seasonal_ensemble/run_ensemble.sh > $LOGS/ensemble_main.log 2>&1 &

# 4. Monitor progress
tail -f $LOGS/ensemble_main.log

# 5. Detach: Ctrl+B, D
# 6. Reconnect: tmux attach -t ensemble
```

### Useful tmux commands
| Command | Description |
|---------|-------------|
| `tmux new -s ensemble` | Create named session |
| `Ctrl+B, D` | Detach (keeps running) |
| `tmux attach -t ensemble` | Reconnect |
| `tmux ls` | List sessions |
| `tmux kill-session -t ensemble` | Kill session |